In [81]:
import sqlite3
import pandas as pd
import os
from core.variables.variable import DATA_DIRECTORY,COLUMNS_STRUCTURE,REVENU,DEPENSE

def explorer_bdd(nom_db: str):
    """
    Affiche le contenu de toutes les tables d'une base de données SQLite.
    Input: nom de la base (ex: "MaCompta_Test")
    """
    # Construction du chemin vers le dossier _data
    path_db = os.path.join(DATA_DIRECTORY, f"{nom_db}.db")
    
    if not os.path.exists(path_db):
        print(f"Erreur : La base de données '{path_db}' est introuvable.")
        return

    try:
        # Connexion à la base
        conn = sqlite3.connect(path_db)
        
        # 1. Lister toutes les tables existantes
        query_tables = "SELECT name FROM sqlite_master WHERE type='table';"
        tables = pd.read_sql(query_tables, conn)['name'].tolist()
        
        if not tables:
            print(f"La base de données '{nom_db}' est vide (aucune table).")
            return

        print(f"=== EXPLORATION DE LA BDD : {nom_db} ===")
        print(f"Tables trouvées : {', '.join(tables)}\n")

        # 2. Afficher le contenu de chaque table
        for table in tables:
            print(f"--- Table : {table} ---")
            df = pd.read_sql(f"SELECT * FROM {table}", conn)
            
            if df.empty:
                print("Table vide.")
            else:
                # On affiche les 10 premières lignes pour ne pas encombrer le notebook
                display(df) # Utilise display() qui est plus propre que print() dans un Notebook
            print("\n")

        conn.close()
        
    except Exception as e:
        print(f"Une erreur est survenue lors de la lecture : {e}")

# --- EXEMPLE D'UTILISATION DANS TON NOTEBOOK ---
explorer_bdd("Compte")

=== EXPLORATION DE LA BDD : Compte ===
Tables trouvées : CH_compte_courant, Compte_Courant, Compte_Jeune, Livret_Bleu, _metadata_accounts

--- Table : CH_compte_courant ---


,Date,Intitule,Categorie,Classe,Type,Valeur
0,01/04/2026,Loyer,Essentiel,Logement,Depense,1400.00
1,2026-03-28,Framboisier + Blue berry,Loisir,Plante,Depense,51.10
2,2026-03-26,Paye,Travail,Paye,Revenu,3385.10
3,2026-03-24,Gala fun,Loisir,special,Depense,91.39
4,24/03/2026,course,Essentiel,Nourriture,Depense,20.85
5,2026-03-16,SBB CHF,Voyage,SBB,Depense,62.00
6,16/03/2026,course,Essentiel,Nourriture,Depense,36.25
7,14/03/2026,course,Essentiel,Nourriture,Depense,19.65
8,2026-03-14,Telephérique mont horn,vacance,Randonée,Depense,130.00
9,2026-03-14,Parking zurich,Voyage,voiture,Depense,31.00




--- Table : Compte_Courant ---


,Date,Intitule,Categorie,Classe,Type,Valeur,real_index
0,02/09/2025,Compte au 09/02/2025,Banque,Solde du mois,Revenu,484.42,0
1,2026-03-29,Maquilage,Essentiel,maquillage,Depense,18.98,None
2,2026-03-23,Train sainté dijon,Voyage,Voiture,Depense,55.90,None
3,2026-03-23,Boucle d oreil,Loisir,Bijoux,Depense,20.10,None
4,2026-03-23,Autoroute,Voyage,Voiture,Depense,18.30,None
...,...,...,...,...,...,...,...
430,2026-04-08,fourniture en tout genre,Essentiel,Fourniture,Depense,4.15,None
431,2026-04-10,Photo remise des diplomes,Special,Cadeau,Depense,12.89,None
432,2026-04-14,PinkMin mania,Loisir,Jeux Vidéo,Depense,5.99,None
433,2026-04-07,virement,Virement,CH_compte_courant,Revenu,2665.00,None




--- Table : Compte_Jeune ---


,Date,Intitule,Categorie,Classe,Type,Valeur,real_index
0,01/07/2026,Interet,Banque,Interet,Revenu,66.60,0
1,2025-02-09,initialisation,Banque,Intialisation,Revenu,1665.15,None




--- Table : Livret_Bleu ---


,Date,Intitule,Categorie,Classe,Type,Valeur,real_index
0,2025-02-25,Virement,Banque,Compte_Courant,Depense,400.00,None
1,14/03/2025,Equilibrage,Banque,Compte_Courant,Depense,100.00,1
2,2025-03-17,renflouement,Banque,Compte_Courant,Revenu,350.00,None
3,2025-03-24,Renflouement projet tenardier,Banque,Compte_Courant,Revenu,70.00,None
4,2025-03-25,Renflouement,Banque,Compte_Courant,Revenu,200.00,None
5,2025-03-29,Plante,Banque,Compte_Courant,Depense,84.00,None
6,2025-05-30,Virement,Banque,Compte_Courant,Revenu,300.00,None
7,2025-06-24,Anniversaire et virement,Banque,Compte_Courant,Revenu,600.00,None
8,2025-06-25,Erreur de compte,Erreur,Erreur,Depense,500.00,None
9,2025-07-17,Virement,Banque,Compte_Courant,Depense,100.00,None




--- Table : _metadata_accounts ---


,account_name,devise,state
0,CH_compte_courant,CHF,1
1,Compte_Courant,€,1
2,Compte_Jeune,€,1
3,Livret_Bleu,€,1


In [82]:
import os
from flask import Flask, render_template, send_from_directory
from core.persistence.database_manager import DatabaseManager
db_manager = DatabaseManager("Compte") 

In [83]:
compta= db_manager.comptabilite

In [84]:
for i in range(len(compta.liste_compte)):
    print(compta.liste_compte[i].account_name)
print(compta.liste_compte[1].account_name)

CH_compte_courant
Compte_Courant
Compte_Jeune
Livret_Bleu
Compte_Courant


In [85]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def generer_predictions_finales(df_raw):
    """
    Analyse complète : Isoler Revenu/Dépense + Fourchettes + Cycles + Retards.
    """
    # 1. Préparation et conversion
    df = df_raw.copy()
    df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)
    df['Mois'] = df['Date'].dt.to_period('M')
    
    aujourdhui = datetime.now()
    nb_mois_total = len(df['Mois'].unique())
    resultats = []

    # 2. On groupe par Type d'abord, puis par Classe et Categorie
    # Cela garantit que les revenus et dépenses ne se mélangent jamais
    groups = df.groupby(['Type', 'Classe', 'Categorie'])

    for (t, classe, cat), group in groups:
        group = group.sort_values('Date')
        
        # --- A. ANALYSE FINANCIÈRE (FOURCHETTES) ---
        # Somme par mois pour ce groupe spécifique
        mensuel = group.groupby('Mois')['Valeur'].sum().abs()
        moyenne = mensuel.mean()
        std_valeur = mensuel.std() if len(mensuel) > 1 else 0

        # Calcul des bornes de la fourchette
        marge = max(moyenne * 0.1, std_valeur)
        f_min = round(max(0, moyenne - marge), 2)
        f_max = round(moyenne + marge, 2)

        # --- B. ANALYSE DU CYCLE (JOUR DU MOIS) ---
        jours_du_mois = group['Date'].dt.day
        jour_moyen = jours_du_mois.mean()
        std_jour = jours_du_mois.std() if len(jours_du_mois) > 1 else 0

        # --- C. CALCUL DES SCORES DE PROBABILITÉ ---
        # 1. Présence (Récurrence sur les mois)
        nb_mois_presence = len(mensuel)
        score_presence = (nb_mois_presence / nb_mois_total) * 100
        
        # 2. Régularité Temporelle (Stabilité du jour)
        score_regularite = max(0, 100 - (std_jour * 12)) if len(group) > 1 else 50
        
        # 3. Stabilité du Montant
        coeff_var = std_valeur / moyenne if moyenne > 0 else 0
        score_montant = max(0, 100 - (coeff_var * 100))

        # Pondération finale
        score_final = (score_presence * 0.6) + (score_regularite * 0.25) + (score_montant * 0.15)

        # --- D. ESTIMATION DATE ET GESTION DU RETARD ---
        derniere_date = group['Date'].max()
        # On projette sur le mois suivant
        prochain_mois = (derniere_date.replace(day=1) + timedelta(days=32)).replace(day=1)
        
        jour_cible = int(round(jour_moyen))
        try:
            date_estimee = prochain_mois.replace(day=jour_cible)
        except ValueError:
            date_estimee = prochain_mois + timedelta(days=jour_cible - 1)

        # Gestion spécifique du retard
        statut = "⏳ À venir"
        if date_estimee < aujourdhui:
            jours_retard = (aujourdhui - date_estimee).days
            score_final = max(0, score_final - (jours_retard * 10))
            statut = "⚠️ En retard"
            
            if score_final == 0:
                statut = "🧊 Cycle rompu"

        # --- E. COMPILATION ---
        resultats.append({
            "Type": t, # "Depense" ou "Revenu" bien distincts
            "Classe": classe,
            "Categorie": cat,
            "Montant_Moyen": round(moyenne, 2),
            "Fourchette_Min": f_min,
            "Fourchette_Max": f_max,
            "Date_Prevue": date_estimee.strftime('%d/%m/%Y'),
            "Probabilite": round(score_final, 1),
            "Statut": statut,
            "Fiabilite": "🔥 Critique" if score_final > 80 else "✅ Fiable" if score_final > 45 else "⚖️ Incertain"
        })

    # 3. Sortie triée par Type puis par Probabilité
    df_res = pd.DataFrame(resultats)
    return df_res.sort_values(by=["Type", "Probabilite"], ascending=[True, False])

# --- TEST ---
ts = generer_predictions_finales(compta.liste_compte[1].df)
print(ts)

       Type         Classe         Categorie  Montant_Moyen  Fourchette_Min  \
35  Depense        Musique            Loisir           7.81            5.77   
36  Depense     Nourriture         Essentiel         213.86           72.65   
37  Depense     Nourriture            Loisir          88.85           27.33   
18  Depense     Cotisation            Banque           5.30            1.33   
31  Depense     Jeux Vidéo            Loisir          26.27            9.61   
..      ...            ...               ...            ...             ...   
82   Revenu  Solde du mois            Banque         484.42          435.98   
83   Revenu  Solde du mois  Erreur de compte          28.03           25.23   
84   Revenu          Train           Special          70.00           63.00   
85   Revenu        Voiture            Poisse         246.00          221.40   
86   Revenu        special            Banque         150.00          135.00   

    Fourchette_Max Date_Prevue  Probabilite        

In [86]:
ts[ts["Type"]=="Revenu"]

,Type,Classe,Categorie,Montant_Moyen,Fourchette_Min,Fourchette_Max,Date_Prevue,Probabilite,Statut,Fiabilite
69,Revenu,CH_compte_courant,Virement,2665.00,2398.50,2931.50,07/05/2026,31.5,⏳ À venir,⚖️ Incertain
78,Revenu,Livret_Bleu,Virement,2070.00,0.00,5025.74,11/05/2026,24.0,⏳ À venir,⚖️ Incertain
65,Revenu,APL,Revenu,196.00,176.40,215.60,05/09/2025,0.0,🧊 Cycle rompu,⚖️ Incertain
66,Revenu,Auto-Entreprenariat,Travail,400.00,360.00,440.00,17/04/2025,0.0,🧊 Cycle rompu,⚖️ Incertain
67,Revenu,Autre,Erreur de compte,64.14,57.73,70.55,16/08/2025,0.0,🧊 Cycle rompu,⚖️ Incertain
68,Revenu,CHF_Compte_Courant,Virement,2155.82,1940.24,2371.40,16/03/2026,0.0,🧊 Cycle rompu,⚖️ Incertain
70,Revenu,Cadeau,Banque,190.00,171.00,209.00,30/01/2026,0.0,🧊 Cycle rompu,⚖️ Incertain
71,Revenu,Cadeau,Special,132.50,93.61,171.39,21/01/2026,0.0,🧊 Cycle rompu,⚖️ Incertain
72,Revenu,Dette,Dette,3300.00,2970.00,3630.00,02/11/2025,0.0,🧊 Cycle rompu,⚖️ Incertain
73,Revenu,Erreur de compte,Banque,80.00,72.00,88.00,23/01/2026,0.0,🧊 Cycle rompu,⚖️ Incertain


In [87]:
def preparer_graphique_prediction(df, solde_actuel):
    # Appeler la fonction de prédiction
    preds = generer_predictions_finales(df)
    
    # On ne garde que les prédictions fiables et à venir
    # On filtre sur "⏳ À venir" ou on peut aussi filtrer par Probabilite > 40
    futures = preds[preds['Statut'] == "⏳ À venir"].copy()
    
    # On convertit les dates pour le tri chronologique
    futures['Date_Prevue_DT'] = pd.to_datetime(futures['Date_Prevue'], format='%d/%m/%Y')
    futures = futures.sort_values('Date_Prevue_DT')
    
    points_graphique = []
    solde_temp = solde_actuel
    
    # Point de départ (Aujourd'hui)
    points_graphique.append({
        "x": datetime.now().strftime('%Y-%m-%d'), 
        "y": round(solde_temp, 2),
        "label": "Solde Actuel"
    })
    
    for _, row in futures.iterrows():
        # Utilisation de Montant_Moyen (nom corrigé)
        valeur = row['Montant_Moyen'] if row['Type'] == 'Revenu' else -row['Montant_Moyen']
        solde_temp += valeur
        
        points_graphique.append({
            "x": row['Date_Prevue_DT'].strftime('%Y-%m-%d'),
            "y": round(solde_temp, 2),
            # On reconstruit le label car 'Prediction' n'existe plus en tant que colonne unique
            "label": f"{row['Categorie']} ({row['Classe']})"
        })
        
    return points_graphique

# Test
_, _, _, stats = compta.liste_compte[1].statistic()
print(preparer_graphique_prediction(compta.liste_compte[1].df, stats))

[{'x': '2026-04-18', 'y': Type               Depense    Revenu     Ecart
Categorie                                     
Banque              302.00   1388.84   1086.84
Dette              3000.00   3300.00    300.00
Erreur de compte    518.68    418.67   -100.01
Essentiel          4001.29      0.00  -4001.29
Habitation        14784.21      0.00 -14784.21
Loisir             2500.54      6.99  -2493.55
Poisse              766.80    246.00   -520.80
Reve                985.41      0.00   -985.41
Revenu                0.00   1176.00   1176.00
Special             564.04   6117.00   5552.96
Travail             785.73  14865.84  14080.11
Vacance             582.76      0.00   -582.76
Virement          14334.00  17240.82   2906.82
Voyage             1452.28      0.00  -1452.28, 'label': 'Solde Actuel'}, {'x': '2026-04-21', 'y': Type               Depense    Revenu     Ecart
Categorie                                     
Banque              294.19   1381.03   1079.03
Dette              2992.19   

In [88]:
import plotly.graph_objects as go
import pandas as pd

def generer_sankey_entonnoir_pur(df_raw,name):
    df = df_raw.copy()
    df['Valeur'] = df['Valeur'].abs()

    # 1. Extraction des deux types
    df_rev = df[df['Type'] == 'Revenu']
    df_dep = df[df['Type'] == 'Depense']

    # --- ÉTAPE 1 : REVENUS -> POINT CENTRAL ---
    flux_entree = df_rev.groupby('Categorie')['Valeur'].sum().reset_index()
    flux_entree.columns = ['source', 'value']
    flux_entree['target'] = 'PORTEFEUILLE'
    # Suffixe pour éviter les collisions de noms entre revenu et dépense
    flux_entree['source'] = flux_entree['source'] + " " 

    # --- ÉTAPE 2 : POINT CENTRAL -> DÉPENSES ---
    flux_sortie = df_dep.groupby('Categorie')['Valeur'].sum().reset_index()
    flux_sortie.columns = ['target', 'value']
    flux_sortie['source'] = 'PORTEFEUILLE'

    # Fusion des flux
    flux_total = pd.concat([flux_entree, flux_sortie], ignore_index=True)

    # 2. Indexation des noeuds
    nodes = list(pd.unique(flux_total[['source', 'target']].values.ravel('K')))
    mapping = {node: i for i, node in enumerate(nodes)}
    
    flux_total['source_idx'] = flux_total['source'].map(mapping)
    flux_total['target_idx'] = flux_total['target'].map(mapping)

    # 3. Couleurs pour un look "Premium Dashboard"
    node_colors = []
    for node in nodes:
        if node == 'PORTEFEUILLE':
            node_colors.append("#00adb5") # Ton cyan fétiche pour le centre
        elif node.endswith(" "):
            node_colors.append("#2ecc71") # Vert pour les Revenus
        else:
            node_colors.append("#e74c3c") # Rouge pour les Dépenses

    # 4. Création du graphique
    fig = go.Figure(data=[go.Sankey(
        arrangement = "snap",
        node = dict(
          pad = 25,
          thickness = 40,
          label = [n.strip() for n in nodes], # On enlève l'espace pour l'affichage
          color = node_colors,
          line = dict(color = "rgba(0,0,0,0)", width = 0)
        ),
        link = dict(
          source = flux_total['source_idx'],
          target = flux_total['target_idx'],
          value = flux_total['value'],
          color = "rgba(144, 148, 151, 0.25)" # Liens discrets
        )
    )])

    fig.update_layout(
        title_text=f"FLUX DE TRÉSORERIE {name}",
        template="plotly_dark",
        font_size=14,
        height=650,
        margin=dict(l=10, r=10, t=50, b=10)
    )
    
    fig.show()
nb = 0
# --- TEST ---
generer_sankey_entonnoir_pur(compta.liste_compte[nb].df,compta.liste_compte[nb].account_name  )